# 03 - TF-IDF/LR and non-learning baselines (P6). Tuned on val_a, then locked.

In [ ]:
# Colab setup: repository at /content/kev-text (git clone or unzip), then run all.
import os, sys
REPO = "/content/kev-text"
if os.path.exists(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
# !pip install -q -r requirements.txt          # once per Colab session
import numpy as np, pandas as pd
from src import config, io_utils, labels, evaluate
SYN = False   # True = dry run on src.synthetic; everything goes to outputs_synthetic/

In [ ]:
if SYN:
    from src import synthetic
    df = synthetic.make(6000)
    if not config.SPLITS_LOCK.exists():                   # mirrors the extras notebook 02 writes
        from src import sensitivity
        sp0 = labels.assign_split(df["published_date"])
        m = sensitivity.kev_retroactive_monthly(df[["cve_id", "kev_date_added"]].dropna(),
                                                df[["cve_id", "published_date"]])
        vb = df[(sp0 == "val") & (df["published_date"] > config.VAL_A_END)]
        labels.write_lock(df, "initial_seal", "synthetic dry run",
                          extra={"s1": sensitivity.s1_record(m, df, sp0),
                                 "conformal_alpha": config.conformal_alpha(int(vb["label_12m"].sum()))})
else:
    df = pd.read_parquet(config.DATA / "dataset_labelled.parquet")
lock = labels.require_lock()
X, y, split = labels.get_xy(df, labels.CLEAN)
split.value_counts()

In [ ]:
from src import models, baselines, seeds
grid = [{"C": c} for c in (0.1, 0.3, 1, 3, 10)]
best, grid_scores = models.tune_on_val_a(models.tfidf_lr, grid, X, y, split)
for seed in config.SEEDS_FINAL:
    seeds.seed_everything(seed)
    with evaluate.log_run("tfidf-lr", seed, "----", is_synthetic=SYN):
        r = models.tfidf_lr(X, y, split, seed=seed, **best)
    evaluate.save_scores("tfidf-lr", seed, r["scores"], is_synthetic=SYN)
evaluate.lock_from_val_a("tfidf-lr", y, best, is_synthetic=SYN)
best, grid_scores

In [ ]:
d = df.loc[X.index]
nl = {"baseline-cvss": baselines.cvss_score(d),
      "baseline-cwe-rate": baselines.group_rate(d, split, "cwe"),
      "baseline-vendor-rate": baselines.group_rate(d, split, "vendor"),
      "baseline-keywords": baselines.keyword_score(X["text"])}
for name, s in nl.items():
    evaluate.save_scores(name, 0, {sp: s[split == sp] for sp in ("val_a", "val_b", "test")}, is_synthetic=SYN)
    evaluate.lock_from_val_a(name, y, {}, is_synthetic=SYN)